In [1]:
import torch
from torchvision import transforms
from PIL import Image
import torch.nn.functional as F
import os
import json
import open_clip
from transformers import CLIPProcessor, CLIPVisionModelWithProjection

In [2]:
# Load the configuration from the JSON file
config_path = "data_config.json"
with open(config_path, "r") as config_file:
    config = json.load(config_file)

# Access the paths from the config
data_path = config["data_path"]
img_directory_training = config["img_directory_training"]
img_directory_test = config["img_directory_test"]
directory = img_directory_training
dirnames = [d for d in os.listdir(directory) if os.path.isdir(os.path.join(directory, d))]
dirnames.sort()
image_folders = dirnames
images = []
raw_image_1 = os.path.join(img_directory_training, "00001_aardvark","aardvark_01b.jpg")
images.append(raw_image_1)
raw_image_2 = os.path.join(img_directory_training, "00001_aardvark","aardvark_02s.jpg")
images.append(raw_image_2)
raw_image_3 = os.path.join(img_directory_training, "00001_aardvark","aardvark_03s.jpg")
images.append(raw_image_3)
raw_image_4 = os.path.join(img_directory_training, "00001_aardvark","aardvark_04s.jpg")
images.append(raw_image_4)

In [6]:
proxy = 'http://127.0.0.1:7897'
os.environ['http_proxy'] = proxy
os.environ['https_proxy'] = proxy
cuda_device_count = torch.cuda.device_count()
device = "cuda:0" if torch.cuda.is_available() else "cpu"
image_encoder = CLIPVisionModelWithProjection.from_pretrained("openai/clip-vit-large-patch14")
image_encoder.requires_grad_(False)
image_encoder = image_encoder.to(device)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")

In [7]:
# version 1
model_type = 'ViT-H-14'
vlmodel, preprocess, feature_extractor = open_clip.create_model_and_transforms(
    model_type, pretrained='laion2b_s32b_b79k', precision='fp32', device = device)
vlmodel.requires_grad_(False)

def Clip_ViT_H_14_ImageEncoder(images):
    # Prevent memory overflow on the GPU
    batch_size = 20
    image_features_list = []

    for i in range(0, len(images), batch_size):
        batch_images = images[i:i + batch_size]
        image_inputs = torch.stack([preprocess(Image.open(img).convert("RGB")) for img in batch_images]).to(
            device)

        with torch.no_grad():
            # vlmodel.encode_image
            batch_image_features = vlmodel.encode_image(image_inputs)
            # batch_image_features /= batch_image_features.norm(dim=-1, keepdim=True)
            batch_image_features = F.normalize(batch_image_features, dim=-1).detach()

        image_features_list.append(batch_image_features)

    clip_image_features = torch.cat(image_features_list, dim=0)

    return clip_image_features

img_clip_latents_ViT_H_14 = Clip_ViT_H_14_ImageEncoder(images)
print(img_clip_latents_ViT_H_14.shape)

torch.Size([4, 1024])


In [3]:
# version 2
proxy = 'http://127.0.0.1:7897'
os.environ['http_proxy'] = proxy
os.environ['https_proxy'] = proxy
cuda_device_count = torch.cuda.device_count()
device = "cuda:0" if torch.cuda.is_available() else "cpu"

# model_type = 'ViT-H-14'
# vlmodel, preprocess_train, feature_extractor = open_clip.create_model_and_transforms(
#     model_type, pretrained='laion2b_s32b_b79k', precision='fp32', device = device)
# vlmodel.requires_grad_(False)

image_encoder = CLIPVisionModelWithProjection.from_pretrained("openai/clip-vit-large-patch14")
image_encoder.requires_grad_(False)
image_encoder = image_encoder.to(device)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-large-patch14")

def ClipImageEncoder(images):
    # Prevent memory overflow on the GPU
    batch_size = 2  
    image_features_list = []
      
    for i in range(0, len(images), batch_size):
        batch_images = images[i:i + batch_size]
        #image_inputs = torch.stack([processor(images = Image.open(img).convert("RGB"), return_tensors="pt").pixel_values[0] for img in batch_images]).to(device)
        image_inputs = processor(images=[Image.open(img).convert("RGB") for img in batch_images],
                         return_tensors="pt").pixel_values.to(device)

        with torch.no_grad():
            # vlmodel.encode_image
            batch_image_features = image_encoder(image_inputs)
            batch_image_features = batch_image_features.image_embeds
            # batch_image_features /= batch_image_features.norm(dim=-1, keepdim=True)
            batch_image_features = F.normalize(batch_image_features, dim=-1).detach()

        image_features_list.append(batch_image_features)

    clip_image_features = torch.cat(image_features_list, dim=0)
        
    return clip_image_features

img_clip_latents = ClipImageEncoder(images)
print(img_clip_latents.shape)

torch.Size([4, 768])


In [4]:
image_encoder.config.projection_dim

768

In [11]:
image_encoder = CLIPVisionModelWithProjection.from_pretrained("openai/clip-vit-large-patch14")
image_encoder.requires_grad_(False)

CLIPVisionModelWithProjection(
  (vision_model): CLIPVisionTransformer(
    (embeddings): CLIPVisionEmbeddings(
      (patch_embedding): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14), bias=False)
      (position_embedding): Embedding(257, 1024)
    )
    (pre_layrnorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-23): 24 x CLIPEncoderLayer(
          (self_attn): CLIPSdpaAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (layer_norm1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=1024, out_fe

In [12]:
from transformers import CLIPTextModelWithProjection

# 加载与 image_encoder 对应的 text_encoder
text_encoder = CLIPTextModelWithProjection.from_pretrained("openai/clip-vit-large-patch14")
text_encoder.requires_grad_(False)

CLIPTextModelWithProjection(
  (text_model): CLIPTextTransformer(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 768)
      (position_embedding): Embedding(77, 768)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPSdpaAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (layer_norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=768, out_features=3072, bias=True)
            (fc2): Linear(in_features=3072, out_features=768, bias=True)
          )
          (layer_norm2): LayerN

In [14]:
# 访问 token_embedding 和 position_embedding
token_embedding = text_encoder.text_model.embeddings.token_embedding
position_embedding = text_encoder.text_model.embeddings.position_embedding

# 打印它们的结构
print("Token Embedding Structure:")
print(token_embedding)

print("\nPosition Embedding Structure:")
print(position_embedding)

Token Embedding Structure:
Embedding(49408, 768)

Position Embedding Structure:
Embedding(77, 768)


In [4]:
from diffusers import AutoencoderKL
vae = AutoencoderKL.from_pretrained("./stable-diffusion-v1-5/", subfolder="vae")
vae.requires_grad_(False)

def VAEImageEncoder(images):
    transform = transforms.Compose([
            transforms.Resize(512, interpolation=transforms.InterpolationMode.BILINEAR),
            transforms.CenterCrop(512),
            transforms.ToTensor(),
            transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
        ])
    batch_size = 20  
    image_emdeddings_list = []
      
    for i in range(0, len(images), batch_size):
        batch_images = images[i:i + batch_size]
        image_inputs = torch.stack([transform(Image.open(img).convert("RGB")) for img in batch_images]).to(device)

        with torch.no_grad():
            batch_image_emdedding = vae.encode(image_inputs).latent_dist.mode() * 0.18215

        image_emdeddings_list.append(batch_image_emdedding)

    image_features = torch.cat(image_emdeddings_list, dim=0)
    return image_features

device = "cuda:0"
vae.to(device)

img_vae_latents = VAEImageEncoder(images)
print(img_vae_latents.shape)

D:\Program\anaconda3\envs\BCI\Lib\site-packages\diffusers\utils\outputs.py:63: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(


torch.Size([4, 4, 64, 64])


In [9]:
class ImageProjModel(torch.nn.Module):
    """Projection Model"""

    def __init__(self, cross_attention_dim=1024, clip_embeddings_dim=1024, clip_extra_context_tokens=4):
        super().__init__()

        self.generator = None
        self.cross_attention_dim = cross_attention_dim
        self.clip_extra_context_tokens = clip_extra_context_tokens
        self.proj = torch.nn.Linear(clip_embeddings_dim, self.clip_extra_context_tokens * cross_attention_dim)
        self.norm = torch.nn.LayerNorm(cross_attention_dim)

    def forward(self, image_embeds):
        embeds = image_embeds
        clip_extra_context_tokens = self.proj(embeds).reshape(
            -1, self.clip_extra_context_tokens, self.cross_attention_dim
        )
        print(f'clip_extra_context_tokens shape is:{clip_extra_context_tokens.shape}')
        clip_extra_context_tokens = self.norm(clip_extra_context_tokens)
        print(f'clip_extra_context_tokens after norm shape is:{clip_extra_context_tokens.shape}')
        return clip_extra_context_tokens

In [5]:
from diffusers import AutoencoderKL, DDPMScheduler, UNet2DConditionModel
unet = UNet2DConditionModel.from_pretrained("./stable-diffusion-v1-5/", subfolder="unet")
unet.requires_grad_(False)

D:\Program\anaconda3\envs\BCI\Lib\site-packages\diffusers\utils\outputs.py:63: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(


UNet2DConditionModel(
  (conv_in): Conv2d(4, 320, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (time_proj): Timesteps()
  (time_embedding): TimestepEmbedding(
    (linear_1): LoRACompatibleLinear(in_features=320, out_features=1280, bias=True)
    (act): SiLU()
    (linear_2): LoRACompatibleLinear(in_features=1280, out_features=1280, bias=True)
  )
  (down_blocks): ModuleList(
    (0): CrossAttnDownBlock2D(
      (attentions): ModuleList(
        (0-1): 2 x Transformer2DModel(
          (norm): GroupNorm(32, 320, eps=1e-06, affine=True)
          (proj_in): LoRACompatibleConv(320, 320, kernel_size=(1, 1), stride=(1, 1))
          (transformer_blocks): ModuleList(
            (0): BasicTransformerBlock(
              (norm1): LayerNorm((320,), eps=1e-05, elementwise_affine=True)
              (attn1): Attention(
                (to_q): LoRACompatibleLinear(in_features=320, out_features=320, bias=False)
                (to_k): LoRACompatibleLinear(in_features=320, out_features=320

In [6]:
unet.config.cross_attention_dim

768

In [34]:
unet.attn_processors.keys()

dict_keys(['down_blocks.0.attentions.0.transformer_blocks.0.attn1.processor', 'down_blocks.0.attentions.0.transformer_blocks.0.attn2.processor', 'down_blocks.0.attentions.1.transformer_blocks.0.attn1.processor', 'down_blocks.0.attentions.1.transformer_blocks.0.attn2.processor', 'down_blocks.1.attentions.0.transformer_blocks.0.attn1.processor', 'down_blocks.1.attentions.0.transformer_blocks.0.attn2.processor', 'down_blocks.1.attentions.1.transformer_blocks.0.attn1.processor', 'down_blocks.1.attentions.1.transformer_blocks.0.attn2.processor', 'down_blocks.2.attentions.0.transformer_blocks.0.attn1.processor', 'down_blocks.2.attentions.0.transformer_blocks.0.attn2.processor', 'down_blocks.2.attentions.1.transformer_blocks.0.attn1.processor', 'down_blocks.2.attentions.1.transformer_blocks.0.attn2.processor', 'up_blocks.1.attentions.0.transformer_blocks.0.attn1.processor', 'up_blocks.1.attentions.0.transformer_blocks.0.attn2.processor', 'up_blocks.1.attentions.1.transformer_blocks.0.attn1.pr

In [35]:
unet.attn_processors.values()

dict_values([<diffusers.models.attention_processor.AttnProcessor2_0 object at 0x00000157DD6DECF0>, <diffusers.models.attention_processor.AttnProcessor2_0 object at 0x00000157DD6DFE60>, <diffusers.models.attention_processor.AttnProcessor2_0 object at 0x00000157DD6DF4D0>, <diffusers.models.attention_processor.AttnProcessor2_0 object at 0x00000157DD6DD070>, <diffusers.models.attention_processor.AttnProcessor2_0 object at 0x00000157DD6DC830>, <diffusers.models.attention_processor.AttnProcessor2_0 object at 0x00000157DD6DC530>, <diffusers.models.attention_processor.AttnProcessor2_0 object at 0x00000157DD6DEF30>, <diffusers.models.attention_processor.AttnProcessor2_0 object at 0x00000157DD6DEB70>, <diffusers.models.attention_processor.AttnProcessor2_0 object at 0x00000157DD6DD5E0>, <diffusers.models.attention_processor.AttnProcessor2_0 object at 0x00000157DD6DCEF0>, <diffusers.models.attention_processor.AttnProcessor2_0 object at 0x00000157DD6CB860>, <diffusers.models.attention_processor.Att

In [33]:
img_proj_model = ImageProjModel(
        cross_attention_dim=unet.config.cross_attention_dim,
        clip_embeddings_dim=image_encoder.config.projection_dim,
        clip_extra_context_tokens=4,
    ).to(device)
image_feature = img_proj_model(img_clip_latents)

clip_extra_context_tokens shape is:torch.Size([4, 4, 768])
clip_extra_context_tokens after norm shape is:torch.Size([4, 4, 768])


In [ ]:
from diffusers import AutoencoderKL, DDPMScheduler, UNet2DConditionModel

In [36]:
unet.config.cross_attention_dim

768

In [37]:
unet.config.block_out_channels[-1]

1280

In [38]:
len("up_blocks.")

10

In [40]:
list(reversed(unet.config.block_out_channels))[1]

1280

In [7]:
import torch.nn as nn
class AttnProcessor(torch.nn.Module):
    r"""
    Processor for implementing scaled dot-product attention (enabled by default if you're using PyTorch 2.0).
    """

    def __init__(
        self,
        hidden_size=None,
        cross_attention_dim=None,
    ):
        super().__init__()
        if not hasattr(F, "scaled_dot_product_attention"):
            raise ImportError("AttnProcessor2_0 requires PyTorch 2.0, to use it, please upgrade PyTorch to 2.0.")

    def __call__(
        self,
        attn,
        hidden_states,
        encoder_hidden_states=None,
        attention_mask=None,
        temb=None,
        *args,
        **kwargs,
    ):
        residual = hidden_states

        if attn.spatial_norm is not None:
            hidden_states = attn.spatial_norm(hidden_states, temb)

        input_ndim = hidden_states.ndim

        if input_ndim == 4:
            batch_size, channel, height, width = hidden_states.shape
            hidden_states = hidden_states.view(batch_size, channel, height * width).transpose(1, 2)

        batch_size, sequence_length, _ = (
            hidden_states.shape if encoder_hidden_states is None else encoder_hidden_states.shape
        )

        if attention_mask is not None:
            attention_mask = attn.prepare_attention_mask(attention_mask, sequence_length, batch_size)
            # scaled_dot_product_attention expects attention_mask shape to be
            # (batch, heads, source_length, target_length)
            attention_mask = attention_mask.view(batch_size, attn.heads, -1, attention_mask.shape[-1])

        if attn.group_norm is not None:
            hidden_states = attn.group_norm(hidden_states.transpose(1, 2)).transpose(1, 2)

        query = attn.to_q(hidden_states)

        if encoder_hidden_states is None:
            encoder_hidden_states = hidden_states
        elif attn.norm_cross:
            encoder_hidden_states = attn.norm_encoder_hidden_states(encoder_hidden_states)

        key = attn.to_k(encoder_hidden_states)
        value = attn.to_v(encoder_hidden_states)

        inner_dim = key.shape[-1]
        head_dim = inner_dim // attn.heads

        query = query.view(batch_size, -1, attn.heads, head_dim).transpose(1, 2)

        key = key.view(batch_size, -1, attn.heads, head_dim).transpose(1, 2)
        value = value.view(batch_size, -1, attn.heads, head_dim).transpose(1, 2)

        # the output of sdp = (batch, num_heads, seq_len, head_dim)
        # TODO: add support for attn.scale when we move to Torch 2.1
        hidden_states = F.scaled_dot_product_attention(
            query, key, value, attn_mask=attention_mask, dropout_p=0.0, is_causal=False
        )

        hidden_states = hidden_states.transpose(1, 2).reshape(batch_size, -1, attn.heads * head_dim)
        hidden_states = hidden_states.to(query.dtype)

        # linear proj
        hidden_states = attn.to_out[0](hidden_states)
        # dropout
        hidden_states = attn.to_out[1](hidden_states)

        if input_ndim == 4:
            hidden_states = hidden_states.transpose(-1, -2).reshape(batch_size, channel, height, width)

        if attn.residual_connection:
            hidden_states = hidden_states + residual

        hidden_states = hidden_states / attn.rescale_output_factor

        return hidden_states

class IPAttnProcessor(torch.nn.Module):
    r"""
    Attention processor for IP-Adapater for PyTorch 2.0.
    Args:
        hidden_size (`int`):
            The hidden size of the attention layer.
        cross_attention_dim (`int`):
            The number of channels in the `encoder_hidden_states`.
        scale (`float`, defaults to 1.0):
            the weight scale of image prompt.
        num_tokens (`int`, defaults to 4 when do ip_adapter_plus it should be 16):
            The context length of the image features.
    """

    def __init__(self, hidden_size, cross_attention_dim=None, scale=1.0, num_tokens=4):
        super().__init__()

        if not hasattr(F, "scaled_dot_product_attention"):
            raise ImportError("AttnProcessor2_0 requires PyTorch 2.0, to use it, please upgrade PyTorch to 2.0.")

        self.hidden_size = hidden_size
        self.cross_attention_dim = cross_attention_dim
        self.scale = scale
        self.num_tokens = num_tokens

        self.to_k_ip = nn.Linear(cross_attention_dim or hidden_size, hidden_size, bias=False)
        self.to_v_ip = nn.Linear(cross_attention_dim or hidden_size, hidden_size, bias=False)

    def __call__(
        self,
        attn,
        hidden_states,
        encoder_hidden_states=None,
        attention_mask=None,
        temb=None,
        *args,
        **kwargs,
    ):
        residual = hidden_states

        if attn.spatial_norm is not None:
            hidden_states = attn.spatial_norm(hidden_states, temb)

        input_ndim = hidden_states.ndim

        if input_ndim == 4:
            batch_size, channel, height, width = hidden_states.shape
            hidden_states = hidden_states.view(batch_size, channel, height * width).transpose(1, 2)

        batch_size, sequence_length, _ = (
            hidden_states.shape if encoder_hidden_states is None else encoder_hidden_states.shape
        )

        if attention_mask is not None:
            attention_mask = attn.prepare_attention_mask(attention_mask, sequence_length, batch_size)
            # scaled_dot_product_attention expects attention_mask shape to be
            # (batch, heads, source_length, target_length)
            attention_mask = attention_mask.view(batch_size, attn.heads, -1, attention_mask.shape[-1])

        if attn.group_norm is not None:
            hidden_states = attn.group_norm(hidden_states.transpose(1, 2)).transpose(1, 2)

        query = attn.to_q(hidden_states)

        if encoder_hidden_states is None:
            encoder_hidden_states = hidden_states
        else:
            # get encoder_hidden_states, ip_hidden_states
            end_pos = encoder_hidden_states.shape[1] - self.num_tokens
            encoder_hidden_states, ip_hidden_states = (
                encoder_hidden_states[:, :end_pos, :],
                encoder_hidden_states[:, end_pos:, :],
            )
            if attn.norm_cross:
                encoder_hidden_states = attn.norm_encoder_hidden_states(encoder_hidden_states)

        key = attn.to_k(encoder_hidden_states)
        value = attn.to_v(encoder_hidden_states)

        inner_dim = key.shape[-1]
        head_dim = inner_dim // attn.heads

        query = query.view(batch_size, -1, attn.heads, head_dim).transpose(1, 2)

        key = key.view(batch_size, -1, attn.heads, head_dim).transpose(1, 2)
        value = value.view(batch_size, -1, attn.heads, head_dim).transpose(1, 2)

        # the output of sdp = (batch, num_heads, seq_len, head_dim)
        # TODO: add support for attn.scale when we move to Torch 2.1
        hidden_states = F.scaled_dot_product_attention(
            query, key, value, attn_mask=attention_mask, dropout_p=0.0, is_causal=False
        )

        hidden_states = hidden_states.transpose(1, 2).reshape(batch_size, -1, attn.heads * head_dim)
        hidden_states = hidden_states.to(query.dtype)

        # for ip-adapter
        ip_key = self.to_k_ip(ip_hidden_states)
        ip_value = self.to_v_ip(ip_hidden_states)

        ip_key = ip_key.view(batch_size, -1, attn.heads, head_dim).transpose(1, 2)
        ip_value = ip_value.view(batch_size, -1, attn.heads, head_dim).transpose(1, 2)

        # the output of sdp = (batch, num_heads, seq_len, head_dim)
        # TODO: add support for attn.scale when we move to Torch 2.1
        ip_hidden_states = F.scaled_dot_product_attention(
            query, ip_key, ip_value, attn_mask=None, dropout_p=0.0, is_causal=False
        )
        with torch.no_grad():
            self.attn_map = query @ ip_key.transpose(-2, -1).softmax(dim=-1)
            #print(self.attn_map.shape)

        ip_hidden_states = ip_hidden_states.transpose(1, 2).reshape(batch_size, -1, attn.heads * head_dim)
        ip_hidden_states = ip_hidden_states.to(query.dtype)

        hidden_states = hidden_states + self.scale * ip_hidden_states

        # linear proj
        hidden_states = attn.to_out[0](hidden_states)
        # dropout
        hidden_states = attn.to_out[1](hidden_states)

        if input_ndim == 4:
            hidden_states = hidden_states.transpose(-1, -2).reshape(batch_size, channel, height, width)

        if attn.residual_connection:
            hidden_states = hidden_states + residual

        hidden_states = hidden_states / attn.rescale_output_factor

        return hidden_states

class IPAdapter(torch.nn.Module):
    """IP-Adapter"""
    def __init__(self, unet, image_proj_model, adapter_modules, ckpt_path=None):
        super().__init__()
        self.unet = unet
        self.image_proj_model = image_proj_model
        self.adapter_modules = adapter_modules

        if ckpt_path is not None:
            self.load_from_checkpoint(ckpt_path)

    def forward(self, noisy_latents, timesteps, encoder_hidden_states, image_embeds):
        ip_tokens = self.image_proj_model(image_embeds)
        encoder_hidden_states = torch.cat([encoder_hidden_states, ip_tokens], dim=1)
        # Predict the noise residual
        noise_pred = self.unet(noisy_latents, timesteps, encoder_hidden_states).sample
        return noise_pred

# init adapter modules
attn_procs = {}
unet_sd = unet.state_dict()
unet_temp = unet
unet_temp.requires_grad_(False)
for name in unet_temp.attn_processors.keys():
    cross_attention_dim = None if name.endswith("attn1.processor") else unet_temp.config.cross_attention_dim
    if name.startswith("mid_block"):
        hidden_size = unet_temp.config.block_out_channels[-1]
    elif name.startswith("up_blocks"):
        block_id = int(name[len("up_blocks.")])
        hidden_size = list(reversed(unet_temp.config.block_out_channels))[block_id]
    elif name.startswith("down_blocks"):
        block_id = int(name[len("down_blocks.")])
        hidden_size = unet_temp.config.block_out_channels[block_id]
    if cross_attention_dim is None:
        attn_procs[name] = AttnProcessor()
    else:
        layer_name = name.split(".processor")[0]
        weights = {
            "to_k_ip.weight": unet_sd[layer_name + ".to_k.weight"],
            "to_v_ip.weight": unet_sd[layer_name + ".to_v.weight"],
        }
        attn_procs[name] = IPAttnProcessor(hidden_size=hidden_size, cross_attention_dim=cross_attention_dim)
        attn_procs[name].load_state_dict(weights)
        #attn_procs[layer_name] = IPAttnProcessor(hidden_size=hidden_size, cross_attention_dim=cross_attention_dim)
        #attn_procs[layer_name].load_state_dict(weights)
unet_temp.set_attn_processor(attn_procs)
adapter_modules = torch.nn.ModuleList(unet_temp.attn_processors.values())

for name, param in unet_temp.named_parameters():
    print(f"{name}: requires_grad={param.requires_grad}")
    
# ip_adapter = IPAdapter(unet_temp, image_proj_model, adapter_modules, args.pretrained_ip_adapter_path)

conv_in.weight: requires_grad=False
conv_in.bias: requires_grad=False
time_embedding.linear_1.weight: requires_grad=False
time_embedding.linear_1.bias: requires_grad=False
time_embedding.linear_2.weight: requires_grad=False
time_embedding.linear_2.bias: requires_grad=False
down_blocks.0.attentions.0.norm.weight: requires_grad=False
down_blocks.0.attentions.0.norm.bias: requires_grad=False
down_blocks.0.attentions.0.proj_in.weight: requires_grad=False
down_blocks.0.attentions.0.proj_in.bias: requires_grad=False
down_blocks.0.attentions.0.transformer_blocks.0.norm1.weight: requires_grad=False
down_blocks.0.attentions.0.transformer_blocks.0.norm1.bias: requires_grad=False
down_blocks.0.attentions.0.transformer_blocks.0.attn1.to_q.weight: requires_grad=False
down_blocks.0.attentions.0.transformer_blocks.0.attn1.to_k.weight: requires_grad=False
down_blocks.0.attentions.0.transformer_blocks.0.attn1.to_v.weight: requires_grad=False
down_blocks.0.attentions.0.transformer_blocks.0.attn1.to_out.

In [44]:
import inspect
print(inspect.getsource(unet_temp.forward))

    def forward(
        self,
        sample: torch.FloatTensor,
        timestep: Union[torch.Tensor, float, int],
        encoder_hidden_states: torch.Tensor,
        class_labels: Optional[torch.Tensor] = None,
        timestep_cond: Optional[torch.Tensor] = None,
        attention_mask: Optional[torch.Tensor] = None,
        cross_attention_kwargs: Optional[Dict[str, Any]] = None,
        added_cond_kwargs: Optional[Dict[str, torch.Tensor]] = None,
        down_block_additional_residuals: Optional[Tuple[torch.Tensor]] = None,
        mid_block_additional_residual: Optional[torch.Tensor] = None,
        down_intrablock_additional_residuals: Optional[Tuple[torch.Tensor]] = None,
        encoder_attention_mask: Optional[torch.Tensor] = None,
        return_dict: bool = True,
    ) -> Union[UNet2DConditionOutput, Tuple]:
        r"""
        The [`UNet2DConditionModel`] forward method.

        Args:
            sample (`torch.FloatTensor`):
                The noisy input tensor with

In [47]:
print(inspect.getsource(unet_temp.set_attn_processor))

    def set_attn_processor(
        self, processor: Union[AttentionProcessor, Dict[str, AttentionProcessor]], _remove_lora=False
    ):
        r"""
        Sets the attention processor to use to compute attention.

        Parameters:
            processor (`dict` of `AttentionProcessor` or only `AttentionProcessor`):
                The instantiated processor class or a dictionary of processor classes that will be set as the processor
                for **all** `Attention` layers.

                If `processor` is a dict, the key needs to define the path to the corresponding cross attention
                processor. This is strongly recommended when setting trainable attention processors.

        """
        count = len(self.attn_processors.keys())

        if isinstance(processor, dict) and len(processor) != count:
            raise ValueError(
                f"A dict of processors was passed, but the number of processors {len(processor)} does not match the"
                f"

In [52]:
from diffusers import AutoencoderKL, DDPMScheduler, UNet2DConditionModel
unet = UNet2DConditionModel.from_pretrained("./stable-diffusion-v1-5/", subfolder="unet")
unet.requires_grad_(False)
processors = unet.attn_processors
for name, processor in processors.items():
    print(f"{name}: {processor}")

down_blocks.0.attentions.0.transformer_blocks.0.attn1.processor: <diffusers.models.attention_processor.AttnProcessor2_0 object at 0x00000157DD5E0980>
down_blocks.0.attentions.0.transformer_blocks.0.attn2.processor: <diffusers.models.attention_processor.AttnProcessor2_0 object at 0x00000157DD5E17F0>
down_blocks.0.attentions.1.transformer_blocks.0.attn1.processor: <diffusers.models.attention_processor.AttnProcessor2_0 object at 0x00000157DD5E1100>
down_blocks.0.attentions.1.transformer_blocks.0.attn2.processor: <diffusers.models.attention_processor.AttnProcessor2_0 object at 0x00000157DD5E10D0>
down_blocks.1.attentions.0.transformer_blocks.0.attn1.processor: <diffusers.models.attention_processor.AttnProcessor2_0 object at 0x00000157DD5E2840>
down_blocks.1.attentions.0.transformer_blocks.0.attn2.processor: <diffusers.models.attention_processor.AttnProcessor2_0 object at 0x00000157DD5E22D0>
down_blocks.1.attentions.1.transformer_blocks.0.attn1.processor: <diffusers.models.attention_process

In [53]:
processors = unet_temp.attn_processors
for name, processor in processors.items():
    print(f"{name}: {processor}")

down_blocks.0.attentions.0.transformer_blocks.0.attn1.processor: AttnProcessor()
down_blocks.0.attentions.0.transformer_blocks.0.attn2.processor: IPAttnProcessor(
  (to_k_ip): Linear(in_features=768, out_features=320, bias=False)
  (to_v_ip): Linear(in_features=768, out_features=320, bias=False)
)
down_blocks.0.attentions.1.transformer_blocks.0.attn1.processor: AttnProcessor()
down_blocks.0.attentions.1.transformer_blocks.0.attn2.processor: IPAttnProcessor(
  (to_k_ip): Linear(in_features=768, out_features=320, bias=False)
  (to_v_ip): Linear(in_features=768, out_features=320, bias=False)
)
down_blocks.1.attentions.0.transformer_blocks.0.attn1.processor: AttnProcessor()
down_blocks.1.attentions.0.transformer_blocks.0.attn2.processor: IPAttnProcessor(
  (to_k_ip): Linear(in_features=768, out_features=640, bias=False)
  (to_v_ip): Linear(in_features=768, out_features=640, bias=False)
)
down_blocks.1.attentions.1.transformer_blocks.0.attn1.processor: AttnProcessor()
down_blocks.1.attentio

In [54]:
from diffusers.models.attention_processor import AttnProcessor2_0
help(AttnProcessor2_0)

Help on class AttnProcessor2_0 in module diffusers.models.attention_processor:

class AttnProcessor2_0(builtins.object)
 |  Processor for implementing scaled dot-product attention (enabled by default if you're using PyTorch 2.0).
 |
 |  Methods defined here:
 |
 |  __call__(self, attn: diffusers.models.attention_processor.Attention, hidden_states: torch.FloatTensor, encoder_hidden_states: Optional[torch.FloatTensor] = None, attention_mask: Optional[torch.FloatTensor] = None, temb: Optional[torch.FloatTensor] = None, scale: float = 1.0) -> torch.FloatTensor
 |      Call self as a function.
 |
 |  __init__(self)
 |      Initialize self.  See help(type(self)) for accurate signature.
 |
 |  ----------------------------------------------------------------------
 |  Data descriptors defined here:
 |
 |  __dict__
 |      dictionary for instance variables
 |
 |  __weakref__
 |      list of weak references to the object

